In [1]:
import os
import pandas as pd
from datetime import date, timedelta, datetime
from sqlalchemy import create_engine, text
from pandas.tseries.offsets import BDay

engine = create_engine('mysql+pymysql://root:@localhost:3306/portfolio_development')
conpf = engine.connect()
engine = create_engine("mysql+pymysql://root:@localhost:3306/stock")
const = engine.connect()
engine = create_engine("sqlite:///c:\\ruby\\port_lite\\db\\development.sqlite3")
conlite = engine.connect()

In [3]:
today = date.today()
# convert the timedelta object to a BusinessDay object
num_business_days = BDay(1)
yesterday = today - num_business_days
yesterday = yesterday.date()
print(f'today: {today}')
print(f'yesterday: {yesterday}')

today: 2026-05-11
yesterday: 2026-05-08


In [5]:
# Get the user's home directory
user_path = os.path.expanduser('~')
# Get the current working directory
current_path = os.getcwd()
# Derive the base directory (base_dir) by removing the last folder ('Daily')
base_path = os.path.dirname(current_path)
#C:\Users\PC1\OneDrive\A5\Data
dat_path = os.path.join(base_path, "Data")
#C:\Users\PC1\Downloads\Datasets
dts_path = os.path.join(user_path, "Downloads", "Datasets")
#C:\Users\PC1\OneDrive\Imports\santisoontarinka@gmail.com - Google Drive\Data>
god_path = os.path.join(user_path, "OneDrive","Imports","santisoontarinka@gmail.com - Google Drive","Data")
#C:\Users\PC1\iCloudDrive\data
icd_path = os.path.join(user_path, "iCloudDrive", "Data")
#C:\Users\PC1\OneDrive\Documents\obsidian-git-sync\Data
osd_path = os.path.join(user_path, "OneDrive","Documents","obsidian-git-sync","Data")
#C:\Users\PC1\OneDrive\A5\Excel
xsl_path = os.path.join(base_path, "Excel")

In [7]:
print("User path:", user_path)
print(f"Current path: {current_path}")
print(f"Base path: {base_path}")
print(f"Data path (dat_path): {dat_path}") 
print(f"Excel path (xsl_path): {xsl_path}") 
print(f"Google Drive path (god_path): {god_path}")
print(f"iCloudDrive path (icd_path): {icd_path}") 
print(f"Obsidian path (osd_path): {osd_path}") 

User path: C:\Users\PC1
Current path: C:\Users\PC1\OneDrive\A4\Daily
Base path: C:\Users\PC1\OneDrive\A4
Data path (dat_path): C:\Users\PC1\OneDrive\A4\Data
Excel path (xsl_path): C:\Users\PC1\OneDrive\A4\Excel
Google Drive path (god_path): C:\Users\PC1\OneDrive\Imports\santisoontarinka@gmail.com - Google Drive\Data
iCloudDrive path (icd_path): C:\Users\PC1\iCloudDrive\Data
Obsidian path (osd_path): C:\Users\PC1\OneDrive\Documents\obsidian-git-sync\Data


In [9]:
format_dict = {
    'shares':'{:,}',    
    'price':'{:.2f}',
    'dividend':'{:.4f}', 
    'date':'{:%Y-%m-%d}', 
    
    'qty':'{:,}','shares':'{:,}',
    'price':'{:.2f}','buy_price':'{:.2f}',
    'dividend':'{:.4f}',    
    'fee':'{:,.2f}','vat':'{:,.2f}','net':'{:,.2f}',
  
    'days':'{:,}',
    'price':'{:.2f}',
    'fee':'{:,.2f}','vat':'{:,.2f}','net':'{:,.2f}','profit':'{:,.2f}',
    'percent':'{:,.2f}%','yearly':'{:,.2f}%',   
    
    'shares':'{:,}',    
    'q4':'{:.4f}','q3':'{:.4f}','q2':'{:.4f}','q1':'{:.4f}','dividend':'{:.4f}',
    'xdate':'{:%Y-%m-%d}','paiddate':'{:%Y-%m-%d}',
    
    'qty':'{:,}','available_qty':'{:,}',
    'cost':'{:.2f}','max_price':'{:.2f}','min_price':'{:.2f}','buy_target':'{:.2f}','sell_target':'{:.2f}',
    'volume':'{:,.2f}','beta':'{:,.2f}'
    }

In [11]:
def show_buy(const, name):
    sql = """
    SELECT * 
    FROM buy 
    WHERE name = '{}'
    """.format(name)
    
    df_buy = pd.read_sql(sql, const)
    df_buy.drop(['volsell', 'volbal','dividend'], axis=1, inplace=True)
    df_buy.rename(columns={'volbuy':'shares'},inplace=True)
    df_buy['shares'] = df_buy['shares'].astype('int64')
    return df_buy.style.format(format_dict).hide(axis="index")

In [13]:
def show_dividend(const, name):
    sql = """
    SELECT * 
    FROM dividend 
    WHERE name = '{}'
    """.format(name)
    
    df_dividend = pd.read_sql(sql, const)
    df_dividend.drop(['PRICE', 'PERCENT'], axis=1, inplace=True)
    df_dividend.columns = df_dividend.columns.str.lower()
    df_dividend['shares'] = df_dividend['shares'].astype('int64')
    df_dividend['xdate'] = pd.to_datetime(df_dividend['xdate'])
    df_dividend['paiddate'] = pd.to_datetime(df_dividend['paiddate'])
    return df_dividend.style.format(format_dict).hide(axis="index")

In [15]:
def show_stocks(conlite, name):
    sql = """
    SELECT * 
    FROM stocks 
    WHERE name = '{}'
    """.format(name)
    
    df_stocks = pd.read_sql(sql, conlite)
    return df_stocks.style.format(format_dict).hide(axis="index")

In [17]:
def update_buy(const, name, new_qty, new_price):
    # Use parameterized query to avoid SQL injection
    sqlUpd = text("""
        UPDATE buy
        SET volbuy = :new_qty,
        price = :new_price
        WHERE name = :name
    """)
    
    # Execute the query with parameters
    result = const.execute(sqlUpd, {
        'new_qty': new_qty,
        'new_price': new_price,
        'name': name
    })

    return f"Records updated = {result.rowcount}"

In [19]:
def update_dividend(const, name, new_qty):
    # Use parameterized query to avoid SQL injection
    sqlUpd = text("""
        UPDATE dividend
        SET shares = :new_qty
        WHERE name = :name
    """)
    
    # Execute the query with parameters
    result = const.execute(sqlUpd, {
        'new_qty': new_qty,  # Use the scalar value
        'name': name
    })

    return f"Records updated = {result.rowcount}"

In [21]:
def update_stock(conlite, name, new_qty, new_unit_cost, new_buy_target, new_sell_target, new_buy_qty):  
    # Use parameterized query to avoid SQL injection
    sqlUpd = text("""
        UPDATE stocks
        SET available_qty = :new_qty, 
            cost = :new_unit_cost,
            buy_target = :new_buy_target,
            sell_target = :new_sell_target,    
            qty = :new_buy_qty
        WHERE name = :name
    """)    
    # Execute the query with parameters
    result = conlite.execute(sqlUpd, {
        'new_qty': new_qty,  # Use the scalar value
        'new_unit_cost': new_unit_cost, # Use the scalar value 
        'new_buy_target': new_buy_target,
        'new_sell_target': new_sell_target,     
        'new_buy_qty': new_buy_qty,
        'name': name
    })
    return f"Records updated = {result.rowcount}"

### End of Definition

## Begin of Sale transaction

In [25]:
# Sells table in MySQL portfolio database
sql = """
SELECT name, stock_id, B.date AS buy_date, qty, B.price AS buy_price, S.* 
FROM sells S
JOIN buys B ON S.buy_id = B.id
JOIN stocks T ON B.stock_id = T.id
ORDER BY S.id DESC
LIMIT 1"""
df_sells_latest = pd.read_sql(sql, conpf)
df_sells_latest.style.format(format_dict).hide(axis="index")

name,stock_id,buy_date,qty,buy_price,id,buy_id,date,price,fee,vat,net,days,profit,percent,yearly,sequence,chart,dividend_id
MCS,43,2026-04-27,"5,000",8.00,1124,1357,2026-05-11,7.80,80.73,5.65,"38,913.62",14,"-1,174.98",-2.93%,-76.41%,-124,None,0


### In case of sale, retreive buy id from sells record (buy_id)

In [28]:
# Buys table in MySQL portfolio database
transaction = 'S'

names = df_sells_latest['name']
name = names.to_string(index=False)

sr_qty = df_sells_latest['qty']
qty = sr_qty.to_string(index=False)
qty = int(qty) * -1

sr_price = df_sells_latest['buy_price']
buy_price = sr_price.to_string(index=False)
buy_price = float(buy_price)

sr_price = df_sells_latest['price']
sell_price = sr_price.to_string(index=False)
sell_price = float(sell_price)

buy_ids = df_sells_latest['buy_id']
buy_id = buy_ids.to_string(index=False)
print(name, qty, buy_price, sell_price)

MCS -5000 8.0 7.8


### Begin of Update buy table in MySQL stock database from sale transaction

In [31]:
sql = f"""
    SELECT *
    FROM buy 
    WHERE name = '{name}'
"""   
df_buy = pd.read_sql(sql, const)
df_buy.drop(['volsell', 'volbal','dividend'], axis=1, inplace=True)
df_buy.rename(columns={'volbuy':'shares'},inplace=True)
df_buy['shares'] = df_buy['shares'].astype('int64')
df_buy.style.format(format_dict).hide(axis="index")

name,date,shares,price,active,period,grade
MCS,2016-09-20,"80,000",14.85,1,2,A4


In [33]:
sql = f"""
    SELECT *
    FROM stocks
    WHERE name = '{name}'
"""
print(sql)
df_tmp = pd.read_sql(sql, conpf)
df_tmp


    SELECT *
    FROM stocks
    WHERE name = 'MCS'



,id,name,category_id,website
0,43,MCS,3,www.mcssteel.com


In [35]:
old_qty = int(df_buy['shares'].iloc[0])
old_price = df_buy['price'].iloc[0]
old_qty, old_price 

(80000, 14.85)

In [37]:
sql = f"""
    SELECT name, B.* 
    FROM buys B
    JOIN stocks T 
    ON B.stock_id = T.id AND status = 'Active'
    WHERE name = '{name}' 
    ORDER BY id DESC
"""
print(sql)
df_buys = pd.read_sql(sql, conpf)
# Format all floats (pandas 1.4.0+)
df_buys.style.format(format_dict).hide(axis="index")


    SELECT name, B.* 
    FROM buys B
    JOIN stocks T 
    ON B.stock_id = T.id AND status = 'Active'
    WHERE name = 'MCS' 
    ORDER BY id DESC



name,id,stock_id,date,qty,price,fee,vat,status,net,kind,chart
MCS,1161,43,2022-04-19,"11,900",12.80,315.30,22.07,Active,"152,657.37",HD,None
MCS,1160,43,2022-04-18,"13,100",12.80,347.10,24.30,Active,"168,051.40",HD,None
MCS,290,43,2016-11-17,"10,000",16.70,345.69,24.20,Active,"167,369.89",HD,None
MCS,289,43,2016-11-14,"10,000",16.70,345.69,24.20,Active,"167,369.89",HD,None
MCS,288,43,2016-10-31,"10,000",16.70,345.69,24.20,Active,"167,369.89",HD,None
MCS,287,43,2016-10-13,"10,000",16.70,345.69,24.20,Active,"167,369.89",HD,None
MCS,286,43,2016-09-20,"10,000",16.70,345.69,24.20,Active,"167,369.89",HD,None


In [39]:
if not df_buys.empty:
    # Calculate total price*qty and total qty
    total_price_qty = (df_buys['price'] * df_buys['qty']).sum()
    total_qty = df_buys['qty'].sum()
    
    # Compute average unit cost
    average_unit_cost = round(total_price_qty / total_qty,2)
else:
    total_qty = 0
    average_unit_cost = 0
    total_price_qty = 0
    
print(f"Total qty: {total_qty:,.0f}")
print(f"Average Unit Cost: {average_unit_cost:.2f}")
print(f"Total price qty: {total_price_qty:.2f}")

Total qty: 75,000
Average Unit Cost: 15.40
Total price qty: 1155000.00


In [41]:
old_cost_amt = df_buy['shares'].iloc[0] * df_buy['price'].iloc[0]
sales_amt = qty * sell_price
new_qty = total_qty
new_cost_amt = average_unit_cost * old_qty
new_unit_cost = average_unit_cost
print(name, old_cost_amt, sales_amt, total_price_qty, new_qty, new_unit_cost)

MCS 1188000.0 -39000.0 1155000.0 75000 15.4


In [43]:
show_buy(const, name)

name,date,shares,price,active,period,grade
MCS,2016-09-20,"80,000",14.85,1,2,A4


In [45]:
print(name, new_qty, new_unit_cost)

MCS 75000 15.4


In [47]:
update_buy(const, name, new_qty, new_unit_cost)

'Records updated = 1'

In [49]:
show_buy(const, name)

name,date,shares,price,active,period,grade
MCS,2016-09-20,"75,000",15.40,1,2,A4


In [51]:
# Define the SQL query
sqlDel = f"""
    DELETE FROM buy
    WHERE name = '{name}' AND volbuy = 0;
"""
print(sqlDel)


    DELETE FROM buy
    WHERE name = 'MCS' AND volbuy = 0;



In [53]:
# Execute the query with the correct parameter dictionary
result = const.execute(text(sqlDel))

# Print the number of rows deleted
print(f"Records deleted: {result.rowcount}")  

Records deleted: 0


### End of Update buy table in MySQL stock database from sale transaction

### Begin of Update dividend table in MySQL stock database from sale transaction

In [57]:
sql = f"""
    SELECT * 
    FROM dividend 
    WHERE name = '{name}'
"""
print(sql)    
df_dividend = pd.read_sql(sql, const)
df_dividend.columns = df_dividend.columns.str.lower()
df_dividend['shares'] = df_dividend['shares'].astype('int64')

df_dividend.style.format(format_dict).hide(axis="index")


    SELECT * 
    FROM dividend 
    WHERE name = 'MCS'



name,q4,q3,q2,q1,dividend,price,percent,shares,xdate,paiddate,kind,actual
MCS,0.7000,0.0000,0.2500,0.0000,0.9500,0.00,0.00%,"80,000",2026-04-20,2026-05-07,,1


In [59]:
new_qty = df_dividend['shares'].iloc[0] + qty
new_qty

75000

In [61]:
show_dividend(const, name)

name,q4,q3,q2,q1,dividend,shares,xdate,paiddate,kind,actual
MCS,0.7000,0.0000,0.2500,0.0000,0.9500,"80,000",2026-04-20,2026-05-07,,1


In [63]:
update_dividend(const, name, new_qty)

'Records updated = 1'

In [65]:
show_dividend(const, name)

name,q4,q3,q2,q1,dividend,shares,xdate,paiddate,kind,actual
MCS,0.7000,0.0000,0.2500,0.0000,0.9500,"75,000",2026-04-20,2026-05-07,,1


In [67]:
# Define the SQL query
sqlDel = f"""
    DELETE FROM dividend
    WHERE name = '{name}' AND shares = 0;
"""

# Execute the query with the correct parameter dictionary
result = const.execute(text(sqlDel))

# Print the number of rows deleted
print(f"Records deleted: {result.rowcount}")  

Records deleted: 0


In [69]:
show_dividend(const, name)

name,q4,q3,q2,q1,dividend,shares,xdate,paiddate,kind,actual
MCS,0.7000,0.0000,0.2500,0.0000,0.9500,"75,000",2026-04-20,2026-05-07,,1


### End of Update dividend table in MySQL stock database from sale transaction

### Begin of Update stocks table in SQLite port_lite database from sale transaction

In [73]:
sql = f"""
    SELECT name, cost, available_qty
    FROM stocks
    WHERE name = '{name}'
"""
print(sql)
df_stocks = pd.read_sql(sql, conlite)
df_stocks['available_qty'] = df_stocks['available_qty'].astype('int64')
df_stocks


    SELECT name, cost, available_qty
    FROM stocks
    WHERE name = 'MCS'



,name,cost,available_qty
0,MCS,14.85,80000


In [75]:
old_qty = int(df_stocks['available_qty'].iloc[0])
new_qty = old_qty + qty
new_buy_target = 0
new_sell_target = 0
new_buy_qty = 0

In [77]:
show_stocks(conlite, name)

id,name,max_price,min_price,status,buy_target,sell_target,volume,beta,cost,qty,buy_spread,sell_spread,available_qty,bl,sh,reason,market
114,MCS,0.00,0.00,S,8.00,8.40,0.00,0.00,14.85,"5,000",-4,4,"80,000",0.000000,0.000000,20Pct,SET


In [79]:
print(name, transaction, new_qty, new_unit_cost, new_buy_target, new_sell_target, new_buy_qty)

MCS S 75000 15.4 0 0 0


In [81]:
update_stock(conlite, name, new_qty, new_unit_cost, new_buy_target, new_sell_target, new_buy_qty)

'Records updated = 1'

In [83]:
show_stocks(conlite, name)

id,name,max_price,min_price,status,buy_target,sell_target,volume,beta,cost,qty,buy_spread,sell_spread,available_qty,bl,sh,reason,market
114,MCS,0.00,0.00,S,0.00,0.00,0.00,0.00,15.40,0,-4,4,"75,000",0.000000,0.000000,20Pct,SET


In [85]:
conlite.commit()

### End of Update stocks table in SQLite port_lite database from sale transaction

## End of Sale process

### =====================================================

In [90]:
#price_date = '2025-02-14'
sql = f"""
    SELECT period, buy.grade AS grade, buy.name AS name, buy.date AS date, 
    FORMAT(volbuy,0) AS volbuy, FORMAT(buy.price,2) AS buy_price, price.price AS mkt_price,
    FORMAT((volbuy * buy.price),2) AS amtbuy, FORMAT((volbuy * price.price),2) AS amtmkt,
    FORMAT(((price.price - buy.price) * volbuy),2) AS amtpol,
    FORMAT((((price.price - buy.price)*volbuy)/(volbuy*buy.price)*100),2) AS pctpol
    FROM buy INNER JOIN price ON buy.name = price.name WHERE price.date = '{yesterday}'
    ORDER BY period, buy.name
"""
print(sql)


    SELECT period, buy.grade AS grade, buy.name AS name, buy.date AS date, 
    FORMAT(volbuy,0) AS volbuy, FORMAT(buy.price,2) AS buy_price, price.price AS mkt_price,
    FORMAT((volbuy * buy.price),2) AS amtbuy, FORMAT((volbuy * price.price),2) AS amtmkt,
    FORMAT(((price.price - buy.price) * volbuy),2) AS amtpol,
    FORMAT((((price.price - buy.price)*volbuy)/(volbuy*buy.price)*100),2) AS pctpol
    FROM buy INNER JOIN price ON buy.name = price.name WHERE price.date = '2026-05-08'
    ORDER BY period, buy.name



In [92]:
sql = f"""
    SELECT period, buy.grade AS grade, buy.name AS name, buy.date AS date, 
    volbuy AS volbuy, buy.price AS buy_price, price.price AS mkt_price,
    (volbuy * buy.price) AS amtbuy, (volbuy * price.price) AS amtmkt, 
    ((price.price - buy.price) * volbuy) AS amtpol, 
    (((price.price - buy.price)*volbuy)/(volbuy*buy.price)*100) AS pctpol 
    FROM buy INNER JOIN price ON buy.name = price.name WHERE price.date = '{yesterday}' 
    ORDER BY period, buy.name
"""
print(sql)
df_buy = pd.read_sql(sql, const)

# Now apply formatting in pandas with correct numeric types
format_dict = {
    'volbuy': '{:,.0f}',
    'buy_price': '{:,.2f}',
    'mkt_price': '{:,.2f}',
    'amtbuy': '{:,.2f}',
    'amtmkt': '{:,.2f}',
    'amtpol': '{:,.2f}',
    'pctpol': '{:,.2f}%'
}

df_buy.style.format(format_dict).hide(axis="index")


    SELECT period, buy.grade AS grade, buy.name AS name, buy.date AS date, 
    volbuy AS volbuy, buy.price AS buy_price, price.price AS mkt_price,
    (volbuy * buy.price) AS amtbuy, (volbuy * price.price) AS amtmkt, 
    ((price.price - buy.price) * volbuy) AS amtpol, 
    (((price.price - buy.price)*volbuy)/(volbuy*buy.price)*100) AS pctpol 
    FROM buy INNER JOIN price ON buy.name = price.name WHERE price.date = '2026-05-08' 
    ORDER BY period, buy.name



period,grade,name,date,volbuy,buy_price,mkt_price,amtbuy,amtmkt,amtpol,pctpol
1,A1,PTTGC,2021-03-17,"6,000",64.75,38.75,"388,500.00","232,500.00","-156,000.00",-40.15%
1,A4,SINGER,2023-01-19,"6,000",24.80,5.80,"148,800.00","34,800.00","-114,000.00",-76.61%
1,C3,STA,2021-06-15,"10,000",30.00,18.90,"300,000.00","189,000.00","-111,000.00",-37.00%
2,A4,3BBIF,2018-05-17,"120,000",10.10,6.60,"1,212,000.00","792,000.00","-420,000.00",-34.65%
2,A4,CPNREIT,2022-08-16,"55,000",18.00,12.30,"990,000.00","676,500.00","-313,500.00",-31.67%
2,A4,DIF,2020-08-01,"45,000",12.70,10.00,"571,500.00","450,000.00","-121,500.00",-21.26%
2,C4,GVREIT,2022-08-24,"69,000",7.75,6.80,"534,750.00","469,200.00","-65,550.00",-12.26%
2,A4,MCS,2016-09-20,"75,000",15.40,8.25,"1,155,000.00","618,750.00","-536,250.00",-46.43%
2,A1,NER,2021-09-01,"27,000",7.45,4.32,"201,150.00","116,640.00","-84,510.00",-42.01%
2,B4,PTT,2025-08-08,"7,500",32.00,36.50,"240,000.00","273,750.00","33,750.00",14.06%


In [94]:
file_name = 'portfolio.csv'
output_file = os.path.join(dat_path, file_name)
god_file = os.path.join(god_path, file_name)
icd_file = os.path.join(icd_path, file_name)
osd_file = os.path.join(osd_path, file_name)

In [96]:
print(f"Output file : {output_file}") 
print(f"icd_file : {icd_file}") 
print(f"god_file : {god_file}") 
print(f"osd_file : {osd_file}") 

Output file : C:\Users\PC1\OneDrive\A4\Data\portfolio.csv
icd_file : C:\Users\PC1\iCloudDrive\Data\portfolio.csv
god_file : C:\Users\PC1\OneDrive\Imports\santisoontarinka@gmail.com - Google Drive\Data\portfolio.csv
osd_file : C:\Users\PC1\OneDrive\Documents\obsidian-git-sync\Data\portfolio.csv


In [98]:
df_buy.to_csv(output_file, header=True, index=False)
df_buy.to_csv(icd_file, header=True, index=False)
df_buy.to_csv(god_file, header=True, index=False)
df_buy.to_csv(osd_file, header=True, index=False)

In [100]:
file_name = '035-portfolio.xlsx'
xsl_file = os.path.join(xsl_path, file_name)
df_buy.to_excel(xsl_file, index=False)